# 03 - Transformer with Defense 1 (Regularization)

Defense: L2 + Dropout + EarlyStopping

In [9]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping

from research_protocol import (
    evaluate_mia_research_protocol,
    compare_defense_vs_baseline,
    set_seed,
)
from research_protocol_robust import evaluate_mia_robust_protocol

In [10]:
ATTACK_SEEDS = [11, 22, 33, 44, 55]

# Attaquant fort (meme base que notebook 02)
N_SHADOWS = 30
SHADOW_EPOCHS = 80
SHADOW_BATCH = 16

ROBUST_N_SHADOWS = 40
ROBUST_SHADOW_EPOCHS = 80
ROBUST_BOUNDARY_DIRS = 32
ROBUST_BOUNDARY_STEPS = 10
ROBUST_BOUNDARY_MAX_ALPHA = 2.0


def build_transformer_base(n_features, dropout=0.15):
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(64)(inp)
    a = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(128, activation='relu')(x)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(64)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m


def build_transformer_reg(n_features, dropout=0.35, l2v=1e-4):
    reg = regularizers.l2(l2v)
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(64, kernel_regularizer=reg)(inp)
    a = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(128, activation='relu', kernel_regularizer=reg)(x)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(64, kernel_regularizer=reg)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m

In [11]:
print('=== Load prepared data ===')
b = np.load('../data_preparation/prepared_oasis2_transformer.npz', allow_pickle=True)
X, y = b['X'], b['y']
X_train, X_test = b['X_train'], b['X_test']
y_train, y_test = b['y_train'], b['y_test']
X_train_seq, X_test_seq = b['X_train_seq'], b['X_test_seq']
X_shadow_raw, y_shadow = b['X_shadow_raw'], b['y_shadow']
SEED = int(b['seed'][0])
print(f'X={X.shape}, train={X_train.shape}, test={X_test.shape}, shadow={X_shadow_raw.shape}')

=== Load prepared data ===
X=(354, 9), train=(70, 9), test=(284, 9), shadow=(107, 9)


In [12]:
print('=== Train Defense 1 model ===')
set_seed(SEED + 10); tf.keras.backend.clear_session()
model_def1 = build_transformer_reg(X_train.shape[1], dropout=0.35, l2v=1e-4)
es = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0)
h = model_def1.fit(X_train_seq, y_train, epochs=140, batch_size=32, validation_split=0.2, callbacks=[es], verbose=0)
p_te = model_def1.predict(X_test_seq, verbose=0).ravel()
test_acc_def1 = accuracy_score(y_test, (p_te >= 0.5).astype(int))
print(f'test_acc_def1={test_acc_def1:.4f}, epochs={len(h.history["loss"])}')

=== Train Defense 1 model ===
test_acc_def1=0.9366, epochs=19


In [13]:
print('=== Run MIA attacks with unified research protocol (strong attacker) ===')

print('Baseline attack-target training (risky)...')
set_seed(SEED + 1)
tf.keras.backend.clear_session()
model_base = build_transformer_base(X_train.shape[1], dropout=0.0)
model_base.fit(X_train_seq, y_train, epochs=260, batch_size=32, verbose=0)
baseline_test_acc = ((model_base.predict(X_test_seq, verbose=0).ravel() >= 0.5).astype(int) == y_test).mean()

baseline_summary, baseline_per_seed = evaluate_mia_research_protocol(
    target_model=model_base,
    X_train_seq=X_train_seq,
    y_train=y_train,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer_base(nf, dropout=0.0),
    seed_list=ATTACK_SEEDS,
    n_shadows=N_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
)

print('Defense 1 target training...')
def1_summary, def1_per_seed = evaluate_mia_research_protocol(
    target_model=model_def1,
    X_train_seq=X_train_seq,
    y_train=y_train,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer_reg(nf, dropout=0.35, l2v=1e-4),
    seed_list=ATTACK_SEEDS,
    n_shadows=N_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
)

print('Baseline summary (mean ± std):')
display(baseline_summary.round(4))
print('Defense 1 summary (mean ± std):')
display(def1_summary.round(4))

=== Run MIA attacks with unified research protocol (strong attacker) ===
Baseline attack-target training (risky)...
Defense 1 target training...
Baseline summary (mean ± std):


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
1,shadow_meta,0.6195,0.0289,0.7099,0.0275,0.2990,0.0398,0.3429,0.0319,0.3187,0.0330
0,logistic,0.4807,0.0143,0.8028,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,threshold_loss,0.4807,0.0143,0.2817,0.0132,0.2154,0.0031,1.0000,0.0000,0.3545,0.0042


Defense 1 summary (mean ± std):


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
1,shadow_meta,0.5298,0.0321,0.7901,0.0059,0.0000,0.0000,0.0000,0.0000,0.0000,0.000
2,threshold_loss,0.5132,0.0230,0.5465,0.0554,0.2126,0.0201,0.4857,0.1302,0.2927,0.043
0,logistic,0.5071,0.0319,0.8028,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000


In [14]:
print('=== Baseline vs Defense 1 (unified protocol) ===')
cmp = compare_defense_vs_baseline(baseline_summary, def1_summary, 'defense1')
display(cmp.round(4))

mcmp = pd.DataFrame([
    {
        'model': 'Transformer',
        'test_acc_baseline': float(baseline_test_acc),
        'test_acc_defense1': float(test_acc_def1),
        'delta_test_acc': float(test_acc_def1 - baseline_test_acc),
    }
])
display(mcmp.round(4))

=== Baseline vs Defense 1 (unified protocol) ===


,attack,auc_mean_baseline,accuracy_mean_baseline,f1_mean_baseline,auc_mean_defense1,accuracy_mean_defense1,f1_mean_defense1,delta_auc,delta_accuracy,delta_f1
1,logistic,0.4807,0.8028,0.0000,0.5071,0.8028,0.0000,0.0264,0.0000,0.0000
0,shadow_meta,0.6195,0.7099,0.3187,0.5298,0.7901,0.0000,-0.0897,0.0803,-0.3187
2,threshold_loss,0.4807,0.2817,0.3545,0.5132,0.5465,0.2927,0.0325,0.2648,-0.0618


,model,test_acc_baseline,test_acc_defense1,delta_test_acc
0,Transformer,0.9225,0.9366,0.0141


In [15]:
print('=== Quick AUC summary ===')
quick_auc = cmp[['attack', 'auc_mean_baseline', 'auc_mean_defense1', 'delta_auc']].sort_values('delta_auc')
display(quick_auc.round(4))

=== Quick AUC summary ===


,attack,auc_mean_baseline,auc_mean_defense1,delta_auc
0,shadow_meta,0.6195,0.5298,-0.0897
1,logistic,0.4807,0.5071,0.0264
2,threshold_loss,0.4807,0.5132,0.0325


In [16]:
print('=== Robust adaptive attack (meta + LiRA + boundary): baseline vs defense1 ===')

base_robust_summary, base_robust_per_seed = evaluate_mia_robust_protocol(
    target_model=model_base,
    x_train_seq=X_train_seq,
    y_train=y_train,
    x_test_seq=X_test_seq,
    y_test=y_test,
    x_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer_base(nf, dropout=0.0),
    seed_list=ATTACK_SEEDS,
    n_shadows=ROBUST_N_SHADOWS,
    shadow_epochs=ROBUST_SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
    boundary_dirs=ROBUST_BOUNDARY_DIRS,
    boundary_steps=ROBUST_BOUNDARY_STEPS,
    boundary_max_alpha=ROBUST_BOUNDARY_MAX_ALPHA,
)

def1_robust_summary, def1_robust_per_seed = evaluate_mia_robust_protocol(
    target_model=model_def1,
    x_train_seq=X_train_seq,
    y_train=y_train,
    x_test_seq=X_test_seq,
    y_test=y_test,
    x_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer_reg(nf, dropout=0.35, l2v=1e-4),
    seed_list=ATTACK_SEEDS,
    n_shadows=ROBUST_N_SHADOWS,
    shadow_epochs=ROBUST_SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
    boundary_dirs=ROBUST_BOUNDARY_DIRS,
    boundary_steps=ROBUST_BOUNDARY_STEPS,
    boundary_max_alpha=ROBUST_BOUNDARY_MAX_ALPHA,
)

cmp_robust = compare_defense_vs_baseline(base_robust_summary, def1_robust_summary, 'defense1')
print('=== Quick AUC summary (robust adaptive) ===')
quick_auc_robust = cmp_robust[['attack', 'auc_mean_baseline', 'auc_mean_defense1', 'delta_auc']].sort_values('delta_auc')
display(quick_auc_robust.round(4))

=== Robust adaptive attack (meta + LiRA + boundary): baseline vs defense1 ===
=== Quick AUC summary (robust adaptive) ===


,attack,auc_mean_baseline,auc_mean_defense1,delta_auc
0,shadow_meta,0.6557,0.5337,-0.122
